# Bonus: give it hands

Eyes, a brain, and now a body. Hold something **red, green or blue** up to the camera
and a robot arm in the MuJoCo physics simulator presses the matching button.

```
 camera ──> which colour? ──> red / green / blue ──> arm presses that button
            (pixels or the                            (MuJoCo, on your laptop)
             vision model)
```

Three colours only, on purpose: the arm has exactly three buttons, and the model is only
*allowed* to answer with one of those three words (or "none"). That constraint is the
most useful trick in this notebook.

Anything solid-coloured works: a sticky note, a mug, a T-shirt, your phone showing a
full-screen colour. No camera? `samples/card_red.png`, `card_green.png`, `card_blue.png`.

In [ ]:
# Run me first (Shift+Enter). Re-run me if you restart the kernel.
%load_ext autoreload
%autoreload 2

import config
from agent import nb, ears, mouth
from agent.llm import LocalLLM, system, user, assistant

llm = LocalLLM()
print("brain:", llm.model_name() if llm.health() else f"NOT RUNNING at {config.SERVER_URL} (run the Step 0 cells in 1_setup.ipynb)")

In [ ]:
import os, time
import cv2
from agent import arm
from agent.eyes import Eyes, preprocess

# "window" opens a real MuJoCo window. "inline" plays the arm as a video right here instead.
VIEW = os.environ.get("AGENT_ARM_VIEW", "window")

## Open MuJoCo

A separate MuJoCo window opens with the arm and three buttons. Drag to orbit, scroll to zoom.
(On a Mac it's launched with `mjpython`, which MuJoCo needs for windows. That's automatic.)
If the window can't open, it falls back to inline.

In [ ]:
if VIEW == "window" and not arm.open_viewer():
    VIEW = "inline"

sim = arm.ArmSim()          # a second copy of the scene, used for pictures and the inline view
nb.show(sim.render(), width=420)
print("buttons:", arm.COLORS, "| view:", VIEW)

## Drive it yourself

No AI yet. Just a function that presses a button. Try all three.

In [ ]:
def press(color):
    if VIEW == "window":
        r = arm.send("tap", color)
        print(f"pressed {color} in {r['seconds']}s")
    else:
        nb.play_frames(sim.tap(color))

press("red")

In [ ]:
press("blue")      # and "green". What happens with "purple"? (try it)

How does it know where to put its hand? No learning: plain geometry. `sim.ik(x, y, z)` turns
a point in space into three joint angles (turn, shoulder, elbow). Peek:

In [ ]:
import numpy as np
for c in arm.COLORS:
    yaw, shoulder, elbow = sim.above(c, 0.05)
    print(f"{c:>5}: turn {np.degrees(yaw):6.1f}°  shoulder {np.degrees(shoulder):6.1f}°  elbow {np.degrees(elbow):6.1f}°")

## Which colour? Two ways to decide

Grab a frame first. Hold something coloured in the middle of the view.

In [ ]:
CAMERA = config.CAMERA          # or "samples/card_green.png"
eyes = Eyes(CAMERA)

frame = eyes.frame()            # re-run this cell after you pick up something else
# frame = cv2.imread("samples/card_green.png")
nb.show(frame, width=360)

### Way 1: count pixels (1990s computer vision)

Convert to HSV colour space, count strongly red / green / blue pixels in the middle of the frame.
Fast, free, and easily fooled by a red wall behind you.

In [ ]:
color, scores = arm.color_by_pixels(frame)
for c, s in scores.items():
    print(f"{c:>5} {s:6.1%} {'#' * int(s * 100)}")
print("->", color)

### Way 2: ask the vision model, but only let it say four words

We pass a JSON schema with `response_format`. llama-server then **constrains** the output
(grammar-based sampling): the model literally can't produce anything except
`{"color": "red" | "green" | "blue" | "none"}`. No parsing surprises, no chatty sentences.

In [ ]:
print(arm.COLOR_SCHEMA)
color, r = arm.color_by_llm(llm, preprocess(frame))
print("model said:", r.text, "|", r.stats())
print("->", color)

## Close the loop: see, decide, press

`watch()` looks every `EVERY` seconds. It only presses when it sees the same new colour
twice in a row, so waving something past the camera doesn't trigger it. Stop with ■.

In [ ]:
DETECTOR = "llm"     # "llm" or "pixels"
EVERY = 1.0          # seconds between looks
SECONDS = 60         # how long to watch

def decide(frame):
    if DETECTOR == "pixels":
        return arm.color_by_pixels(frame)[0]
    return arm.color_by_llm(llm, preprocess(frame))[0]

def watch(seconds=SECONDS):
    last_pressed, candidate, t_end = None, None, time.time() + seconds
    try:
        while time.time() < t_end:
            if not eyes.is_live:
                eyes.next_file()
            t0 = time.perf_counter()
            seen = decide(eyes.frame())
            took = time.perf_counter() - t0
            print(f"sees {str(seen):>5} ({took:.2f}s)")
            if seen and seen == candidate and seen != last_pressed:
                print(f"   -> pressing {seen}")
                mouth.speak(f"{seen}!", block=False)
                press(seen)
                last_pressed = seen
            candidate = seen
            time.sleep(max(0, EVERY - took))
    except KeyboardInterrupt:
        pass
    print("stopped watching")

watch()

## Try this

1. **Race.** Run `watch()` with `DETECTOR = "pixels"` then `"llm"`. Which reacts faster? Which gets fooled by your shirt?
2. **Bad lighting.** Dim the room or tilt the object. Who copes better?
3. **The Stroop test.** Hold up something green with the word **RED** written on it.
   Pixels say green. What does the model say? Change `arm.COLOR_PROMPT` to make it follow the word instead.
4. **A fourth colour.** The limit of three lives in two places: `arm.COLORS` (and `RGBA`, `ANGLE_DEG`) in
   `agent/arm.py`, and the schema. Add yellow at 90°. Restart the kernel and the window.

## Build: press it by voice

"Press the blue one." Record, transcribe, pull the colour out, press. Fill the TODO.

In [ ]:
def colour_from_speech(text: str):
    # TODO: return "red", "green" or "blue" if the sentence mentions one, else None
    return None

audio = nb.record(4)
said, _ = ears.transcribe(audio)
print("heard:", said)
c = colour_from_speech(said)
print("->", c)
if c:
    press(c)

*Solution below: click `...` to reveal.*

In [ ]:
# SOLUTION
def colour_from_speech(text: str):
    words = text.lower().replace(".", " ").replace(",", " ").split()
    return next((c for c in arm.COLORS if c in words), None)

print(repr(said), "->", colour_from_speech(said))

## Build: a sequence

"Press red, then blue, then red again." Let the model turn a sentence into a *list* of
button presses. The schema is an array of the same three words. Fill the TODO.

In [ ]:
SEQUENCE_SCHEMA = {
    "type": "object",
    "properties": {
        # TODO: "sequence": an array (maxItems 5) whose items are one of arm.COLORS
    },
    "required": ["sequence"],
}
REQUEST = "Press red, then blue, then red again."

*Solution below: click `...` to reveal.*

In [ ]:
# SOLUTION
import json
SEQUENCE_SCHEMA = {
    "type": "object",
    "properties": {"sequence": {"type": "array", "maxItems": 5,
                                "items": {"type": "string", "enum": list(arm.COLORS)}}},
    "required": ["sequence"],
}
r = llm.chat([user(f"Turn this into a list of button presses: {REQUEST}")], stream=False, temperature=0,
             max_tokens=60, response_format={"type": "json_schema",
                                             "json_schema": {"name": "sequence", "schema": SEQUENCE_SCHEMA}})
seq = json.loads(r.text)["sequence"]
print(seq)
for c in seq:
    press(c)

## Where this goes next

The arm here follows hand-written geometry. Real robots increasingly swap that for a *learned
policy*: a model that takes camera frames and outputs joint commands directly (LeRobot's ACT,
for example, trains exactly this kind of arm in MuJoCo). Same loop you just built:
see, decide, move. The decision just gets richer.

When you're done:

In [ ]:
if VIEW == "window":
    arm.close_viewer()
eyes.close()